In [10]:
import urllib.request
import os

# 先把 download_reference_model.sh 里的链接填进来
url = "https://www.dropbox.com/scl/fi/ng66ceortfy07bgie3r54/alltracker_reference.tar.gz?rlkey=o781im2v0sl7035hy8fcuv1d5&st=u5mcttcx&dl=1"
save_path = "./alltracker_reference.tar.gz"

print("开始下载...")
urllib.request.urlretrieve(url, save_path)

size = os.path.getsize(save_path)
print(f"✅ 下载完成，文件大小: {size/1024/1024:.1f} MB")

开始下载...
✅ 下载完成，文件大小: 58.4 MB


In [33]:
import sys


import torch
import numpy as np
import cv2
import json
import os
%cd alltracker/
from nets.alltracker import Net as AllTracker

# ============================================================
# 加载模型
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AllTracker(seqlen=16).to(device)
# 自动从 huggingface 下载权重（第一次运行需要联网）
checkpoint = torch.load('./checkpoints/model-000600000.pth', map_location=device)
print(checkpoint.keys())
%cd ..
print(type(checkpoint))
print(checkpoint.keys())
model.load_state_dict(checkpoint['model'])
model.eval()
print("✅ AllTracker 模型加载完成")

# ============================================================
# 读取视频（和 CoTracker 相同）
# ============================================================
def read_video_for_tracker(mp4_path, max_frames=50):
    cap = cv2.VideoCapture(mp4_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"无法打开视频: {mp4_path}")
    frames = []
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    print(f"读取 {len(frames)} 帧，分辨率 {frames[0].shape}")
    return torch.tensor(np.array(frames)).permute(0,3,1,2).unsqueeze(0).float()

video_path = "./dataset/dynamic_seq_000.mp4"
MAX_FRAMES  =16
video_tensor = read_video_for_tracker(video_path, max_frames=MAX_FRAMES).to(device)
B, T, C, H, W = video_tensor.shape

# ============================================================
# 读取标注点
# ============================================================
with open("annotation_003.json", "r") as f:
    ann = json.load(f)
f0_pts = ann["f0"]    # [[x1,y1], ...]
fL_pts = ann["fLast"]
N = len(f0_pts)

# AllTracker queries 格式: [B, N, 3]，每行 [t, y, x]（注意是 y,x 顺序）
queries = torch.tensor(
    [[0.0, y, x] for x, y in f0_pts],  # ✅ AllTracker 用 y,x 顺序
    dtype=torch.float32
).unsqueeze(0).to(device)  # [1, N, 3]

# ============================================================
# 推理
# ============================================================
torch.cuda.empty_cache()
with torch.no_grad():
    # AllTracker 输出: trajs [B,T,N,2], vis [B,T,N], conf [B,T,N]
    trajs, vis, conf = model(video_tensor, queries=queries)

print(f"trajs shape: {trajs.shape}")   # [1, T, N, 2]
print(f"vis shape:   {vis.shape}")     # [1, T, N]
print(f"conf shape:  {conf.shape}")    # [1, T, N]

# ============================================================
# 可视化：用 CoTracker 的 Visualizer（格式兼容）
# 或用 AllTracker 自带的 demo 风格
# ============================================================
os.makedirs("./vis_output", exist_ok=True)

# 方案A：复用 CoTracker Visualizer（格式完全兼容）
from cotracker.utils.visualizer import Visualizer
vis_tool = Visualizer(save_dir="./vis_output", pad_value=120, linewidth=2, fps=10)
vis_tool.visualize(
    video=video_tensor,
    tracks=trajs,
    visibility=vis,
    filename="dynamic_seq_003_alltracker"
)
print("✅ 轨迹视频 → ./vis_output/dynamic_seq_003_alltracker.mp4")

# ============================================================
# 评估（和 CoTracker 完全相同的逻辑）
# ============================================================
def tapvid_delta_avg(pred_last, gt_last, pred_vis_last, video_H, video_W):
    diag = np.sqrt(video_H**2 + video_W**2)
    thresholds = np.array([1, 2, 4, 8, 16]) / 256.0 * diag
    visible_mask = pred_vis_last > 0.5
    if visible_mask.sum() == 0:
        print("⚠️  所有点不可见")
        return None
    errors = np.linalg.norm(pred_last[visible_mask] - gt_last[visible_mask], axis=1)
    accuracies = [(errors < t).mean() for t in thresholds]
    return {
        "delta_avg_vis": np.mean(accuracies),
        "per_threshold": dict(zip([1,2,4,8,16], accuracies)),
        "ATE": errors.mean(),
        "median_error": np.median(errors),
        "num_visible": int(visible_mask.sum()),
    }

pred_last     = trajs[0, -1].cpu().numpy()       # [N, 2]
gt_last       = np.array(fL_pts, dtype=np.float32)
pred_vis_last = vis[0, -1].cpu().numpy()         # [N]
pred_conf_last = conf[0, -1].cpu().numpy()       # [N] AllTracker 独有

print("\n各点可见性 & 置信度（最后一帧）:")
for i, (v, c) in enumerate(zip(pred_vis_last, pred_conf_last)):
    print(f"  点{i+1}: vis={v:.3f}  conf={c:.3f}  {'✅' if v > 0.5 else '❌'}")

results = tapvid_delta_avg(pred_last, gt_last, pred_vis_last, H, W)

if results:
    print(f"\n{'='*50}")
    print(f"       AllTracker 评估结果（seq_003）")
    print(f"{'='*50}")
    print(f"δ_avg^vis:    {results['delta_avg_vis']*100:.1f}%")
    print(f"ATE:          {results['ATE']:.2f} px")
    print(f"中位误差:     {results['median_error']:.2f} px")
    print(f"可见点数:     {results['num_visible']} / {N}")
    print("-" * 50)
    for thresh, acc in results["per_threshold"].items():
        bar = "█" * int(acc * 20)
        print(f"  δ<{thresh:>2}px: {acc*100:5.1f}%  {bar}")

# 最后一帧对比图
last_frame_vis = video_tensor[0, -1].permute(1,2,0).cpu().numpy().astype(np.uint8)
last_frame_vis = cv2.cvtColor(last_frame_vis, cv2.COLOR_RGB2BGR)
for i, ((px,py),(gx,gy)) in enumerate(zip(pred_last, gt_last)):
    cv2.circle(last_frame_vis, (int(gx),int(gy)), 6, (0,255,0), 2)
    cv2.circle(last_frame_vis, (int(px),int(py)), 4, (0,0,255), -1)
    cv2.line(last_frame_vis, (int(gx),int(gy)), (int(px),int(py)), (0,255,255), 1)
    cv2.putText(last_frame_vis, f"P{i+1}", (int(px)+6,int(py)-6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,0,255), 1)
cv2.imwrite("./vis_output/alltracker_last_frame.png", last_frame_vis)
print("✅ 对比图 → ./vis_output/alltracker_last_frame.png")

[Errno 2] No such file or directory: 'alltracker/'
/Users/cx/Desktop


FileNotFoundError: [Errno 2] No such file or directory: './checkpoints/model-000600000.pth'